Note, to run this notebook you need to slightly change the format for starting a coiled session:
> `AWS_ACCESS_KEY_ID="" AWS_SECRET_ACCESS_KEY="" AWS_SESSION_TOKEN="" AWS_PROFILE="" uv run coiled notebook start --vm-type r8g.8xlarge --sync --sync-ignore .venv`

In [1]:
import icechunk

from srm.config import _icechunk_storage_for_path
from srm.qa_flags import (
    ATTRS_TIME_INVARIANT,
    ATTRS_TIME_VARYING,
    FLAG_LIST_TIME_INVARIANT,
    FLAG_LIST_TIME_VARYING,
    combine_intermediate_flags,
    discover_leaves,
    parse_tag,
    write_final_qa_flags,
)

# A. Define what data arrays exist to traverse

In [2]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]  # , "hurs"]

In [3]:
GCMS = ["CESM2-WACCM"]

# BRANCH = "full-regional-run-issue-534"
# ROOT_DIR = "s3://carbonplan-scratch/srm/output/qa/"
# STORE_SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)
# STORE_SUBSET_ID = ArtifactCache._get_subset_id(STORE_SUBSET_BOUNDS)

BRANCH = "v0.13.0"
ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
STORE_SUBSET_ID = "global"

In [4]:
[_, tags, _, _, _, tags_np, _, _] = discover_leaves(
    gcms=GCMS, branch=BRANCH, root_dir=ROOT_DIR, store_subset_id=STORE_SUBSET_ID
)

opened 1/1 stores on branch 'v0.13.0': CESM2-WACCM
86 leaves across 1 GCMs


In [9]:
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags/v0.13.0-global"

# Write out final overall flags

In [10]:
tag = "CESM2-WACCM_rsds_ssp245_003"

[overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
    tag=tag,
    flag_list_time_varying=FLAG_LIST_TIME_VARYING,
    flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
    bucket=BUCKET,
    prefix=PREFIX,
)

lat = overall_flag_time_invariant.lat
lon = overall_flag_time_invariant.lon

In [11]:
print(len(tags))
# This loop takes about 15 minutes to run on v0.13.0 (31 global data arrays)
OVERWRITE = False

repos: dict[str, icechunk.Repository] = {}
for gcm in GCMS:
    repos[gcm] = icechunk.Repository.open(
        _icechunk_storage_for_path(f"{ROOT_DIR}{gcm}-ERA5-{STORE_SUBSET_ID}.icechunk")
    )

for i, tag in enumerate(tags):
    print(tag)
    [gcm, var, scenario, ens] = parse_tag(tag)
    print("calculating flags")
    [overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
        tag=tag,
        flag_list_time_varying=FLAG_LIST_TIME_VARYING,
        flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
        bucket=BUCKET,
        prefix=PREFIX,
    )

    group = f"{scenario}/{var}/{ens}"
    session = repos[gcm].writable_session(BRANCH)

    print("  writing flag_time_varying")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_varying,
        flag_name="flag_time_varying",
        attrs=ATTRS_TIME_VARYING,
        overwrite=OVERWRITE,
    )

    print("  writing flag_time_invariant")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_invariant,
        flag_name="flag_time_invariant",
        attrs=ATTRS_TIME_INVARIANT,
        overwrite=OVERWRITE,
    )

    commit = session.commit(f"write qa flags for {tag}")
    print(f"    commit {commit}")

31
CESM2-WACCM_pr_g6_1p5k_002
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit XFHMTMMMBK0F6SY6GMGG
CESM2-WACCM_pr_g6_1p5k_003
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 0VPKWRPWSRQEZG3BH3Z0
CESM2-WACCM_rsds_g6_1p5k_002
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 26DSY48CZDS44RQJWKBG
CESM2-WACCM_rsds_g6_1p5k_003
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit Z2FW3G1DZ48QT2PRH5Q0
CESM2-WACCM_tas_g6_1p5k_002
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 5Q7CZMCWFMNC5QABFDV0
CESM2-WACCM_tas_g6_1p5k_003
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 5C2N6HHV7JABJRKAX7D0
CESM2-WACCM_tasmin_g6_1p5k_002
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 12YG8SJ5W3GX34KBSTHG
CESM2-WACCM_tasmin_g6_1p5k_003
calculating